[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-11-model-training.ipynb#scrollTo=1a2b3c4d)

---
# Day 11 · Model Training and Experiment Tracking
**certified-journeys / metaflow-certified** · Practice · Model Training

> **Goal for today:** Build a Metaflow training flow that parallelizes hyperparameter search with `@foreach`, collects metrics in a join step, selects the best model, and visualizes results with `@card`.


In [ ]:
%pip install -q metaflow scikit-learn pandas numpy


## Step 1 · The @foreach + join Pattern for Hyperparameter Search

Metaflow's `@foreach` fan-out runs one child branch per element of a list. A subsequent `join` step collects all results. This is the idiomatic pattern for parallelized hyperparameter search.

```
start
  │
  ├─ train(hp=A) ─┐
  ├─ train(hp=B) ─┤  ← @foreach fans out
  └─ train(hp=C) ─┘
                   │
                  join  ← collects all branches
                   │
                  end
```

| Pattern | How |
|---|---|
| Fan-out | `self.next(self.train_model, foreach='hyperparams')` |
| Branch input | `self.hp = self.input` in the foreach step |
| Join | `def join(self, inputs):` — `inputs` is a list of branch task data |
| Collect results | `[inp.metric for inp in inputs]` |


In [ ]:
# Verify all dependencies are importable
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

iris = load_iris(as_frame=True)
df = iris.frame.rename(columns={'target': 'species'})
print(f"Iris dataset ready: {df.shape}")
print(f"Class distribution:\n{df['species'].value_counts().to_dict()}")


**What just happened?**

- All sklearn imports succeeded — we can use them inside Metaflow steps.
- The iris dataset is perfectly balanced (50 samples per class) — ideal for a classification demo.
- **RandomForest** and **LogisticRegression** will be our two model families; we'll sweep hyperparameters for each.


## Step 2 · Building the Training Flow with @foreach

The key mechanics to know:
- `foreach='attr_name'` — `attr_name` must be a list attribute set on `self` in the step that calls `self.next`.
- In the foreach step, `self.input` contains the current element.
- The join step signature is always `def join(self, inputs)` — `inputs` is an iterator of task data objects.
- You **must** call `self.merge_artifacts(inputs)` if you want artifacts from branches to propagate; otherwise access them explicitly via `inputs`.


In [ ]:
%%writefile training_flow.py
from metaflow import FlowSpec, step
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler


class ModelTrainingFlow(FlowSpec):
    """Hyperparameter search via @foreach + join, with best model selection."""

    @step
    def start(self):
        """Prepare data and define the hyperparameter grid."""
        iris = load_iris(as_frame=True)
        df = iris.frame.rename(columns={'target': 'species'})

        X = df.drop(columns='species').values
        y = df['species'].values

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )

        # Scale features — fit on train, apply to test
        scaler = StandardScaler()
        self.X_train = scaler.fit_transform(X_train)
        self.X_test  = scaler.transform(X_test)
        self.y_train = y_train
        self.y_test  = y_test
        self.scaler  = scaler

        # Hyperparameter grid — each dict is one branch in the foreach
        self.hyperparams = [
            {'model': 'rf',  'n_estimators': 10,  'max_depth': 3},
            {'model': 'rf',  'n_estimators': 50,  'max_depth': 5},
            {'model': 'rf',  'n_estimators': 100, 'max_depth': None},
            {'model': 'lr',  'C': 0.1,  'max_iter': 200},
            {'model': 'lr',  'C': 1.0,  'max_iter': 200},
            {'model': 'lr',  'C': 10.0, 'max_iter': 500},
        ]

        print(f"Train: {self.X_train.shape}, Test: {self.X_test.shape}")
        print(f"Searching over {len(self.hyperparams)} configurations")
        # foreach fans out — one branch per element of self.hyperparams
        self.next(self.train_model, foreach='hyperparams')

    @step
    def train_model(self):
        """Train a single model configuration — runs in parallel for each branch."""
        hp = self.input  # the current hyperparameter dict
        self.hp = hp

        if hp['model'] == 'rf':
            clf = RandomForestClassifier(
                n_estimators=hp['n_estimators'],
                max_depth=hp['max_depth'],
                random_state=42
            )
        else:  # lr
            clf = LogisticRegression(
                C=hp['C'],
                max_iter=hp['max_iter'],
                random_state=42
            )

        clf.fit(self.X_train, self.y_train)
        preds = clf.predict(self.X_test)
        self.accuracy = float(accuracy_score(self.y_test, preds))
        self.model    = clf  # persist the fitted model

        print(f"[{hp}] accuracy = {self.accuracy:.4f}")
        self.next(self.join_results)

    @step
    def join_results(self, inputs):
        """Collect metrics from all branches and select the best model."""
        # Gather (accuracy, model, hp) tuples from every branch
        results = [
            (inp.accuracy, inp.model, inp.hp)
            for inp in inputs
        ]

        # Sort by accuracy descending
        results.sort(key=lambda x: x[0], reverse=True)

        self.best_accuracy = results[0][0]
        self.best_model    = results[0][1]
        self.best_hp       = results[0][2]

        # Store all results as a DataFrame for downstream analysis
        import pandas as pd
        self.results_df = pd.DataFrame(
            [(r[2], r[0]) for r in results],
            columns=['hyperparams', 'accuracy']
        )

        # merge_artifacts propagates shared artifacts (X_test, y_test) from branches
        self.merge_artifacts(inputs, include=['X_test', 'y_test', 'scaler'])

        print(f"\nBest config: {self.best_hp}")
        print(f"Best accuracy: {self.best_accuracy:.4f}")
        self.next(self.end)

    @step
    def end(self):
        """Summarise the experiment and confirm best model is persisted."""
        print("\n=== Experiment Summary ===")
        print(self.results_df.to_string(index=False))
        print(f"\nWinner: {self.best_hp}")
        print(f"Best accuracy: {self.best_accuracy:.4f}")
        print("\nBest model stored as self.best_model — ready for inference.")


if __name__ == '__main__':
    ModelTrainingFlow()


**What just happened?**

- We wrote a complete training flow with `@foreach` over a 6-entry hyperparameter grid.
- **`self.hyperparams`** is the list that drives the fan-out; in each branch `self.input` is one element.
- The `join_results` step receives all branches as `inputs` — we collect accuracy scores, sort them, and store the winner.
- **`self.merge_artifacts(inputs, include=[...])`** propagates specific artifacts (X_test, scaler) from branches to the join step — without this, you'd have to re-read them from `inputs[0]`.


In [ ]:
# Run the training flow — watch all 6 branches execute
!python training_flow.py run


**What just happened?**

- Metaflow ran all 6 training branches (sequentially in local mode; in parallel on a remote backend).
- Each `train_model` branch printed its accuracy, then `join_results` sorted and selected the winner.
- **The best model is now a named artifact** — any downstream inference flow can load it without re-training.
- On a remote backend (AWS Batch, Kubernetes), all 6 branches would run simultaneously, making this trivially scalable to 600+ hyperparameter combinations.


## Step 3 · Accessing Results via the Client API

After the training run, use the Client API to retrieve the best model and evaluate it further — or compare across multiple runs (experiment tracking).


In [ ]:
from metaflow import Flow
import numpy as np
from sklearn.metrics import classification_report

# Load the latest training run
run = Flow('ModelTrainingFlow').latest_run
print(f"Run ID: {run.id}  |  Successful: {run.successful}")

end_data = run['end'].task.data

print(f"\nBest hyperparams: {end_data.best_hp}")
print(f"Best accuracy:    {end_data.best_accuracy:.4f}")

# Retrieve model and test data for detailed evaluation
best_model = end_data.best_model
X_test     = end_data.X_test
y_test     = end_data.y_test

preds = best_model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, preds, target_names=['setosa', 'versicolor', 'virginica']))


**What just happened?**

- We retrieved the best model from the run's artifacts and ran a detailed classification report — all without re-training.
- **This is the core of Metaflow experiment tracking**: every run is immutable and queryable. Compare `run1['end'].task.data.best_accuracy` vs `run2['end'].task.data.best_accuracy` to track improvements over time.
- The `X_test` and `y_test` are also stored — you can audit exactly what data the model was evaluated on for each run.


## Step 4 · Using @card for Visualization

`@card` attaches a rich HTML report to a step. Metaflow cards support:
- **Default card**: auto-generated step summary
- **`MetaflowCard` components**: `Markdown`, `Table`, `Artifact` for custom content
- **`@card(type='blank')`**: start from an empty card and build it programmatically

Cards are viewed with `python flow.py card view <run_id>/<step_name>` or in the Metaflow GUI.


In [ ]:
%%writefile training_flow_cards.py
from metaflow import FlowSpec, step, card
from metaflow.cards import Markdown, Table
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler


class ModelTrainingWithCardsFlow(FlowSpec):
    """Training flow with @card reports on the join step."""

    @step
    def start(self):
        iris = load_iris(as_frame=True)
        df = iris.frame.rename(columns={'target': 'species'})
        X = df.drop(columns='species').values
        y = df['species'].values

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        scaler = StandardScaler()
        self.X_train = scaler.fit_transform(X_train)
        self.X_test  = scaler.transform(X_test)
        self.y_train = y_train
        self.y_test  = y_test

        self.hyperparams = [
            {'model': 'rf',  'n_estimators': 10,  'max_depth': 3},
            {'model': 'rf',  'n_estimators': 100, 'max_depth': None},
            {'model': 'lr',  'C': 1.0,  'max_iter': 200},
            {'model': 'lr',  'C': 10.0, 'max_iter': 500},
        ]
        self.next(self.train_model, foreach='hyperparams')

    @step
    def train_model(self):
        hp = self.input
        self.hp = hp
        if hp['model'] == 'rf':
            clf = RandomForestClassifier(
                n_estimators=hp['n_estimators'],
                max_depth=hp['max_depth'],
                random_state=42
            )
        else:
            clf = LogisticRegression(C=hp['C'], max_iter=hp['max_iter'], random_state=42)

        clf.fit(self.X_train, self.y_train)
        preds = clf.predict(self.X_test)
        self.accuracy = float(accuracy_score(self.y_test, preds))
        self.model = clf
        print(f"[{hp}] accuracy = {self.accuracy:.4f}")
        self.next(self.join_results)

    @card(type='blank')  # blank card — we populate it manually
    @step
    def join_results(self, inputs):
        """Join, select best, and attach an experiment-tracking card."""
        from metaflow.cards import current

        results = [(inp.accuracy, inp.model, inp.hp) for inp in inputs]
        results.sort(key=lambda x: x[0], reverse=True)

        self.best_accuracy = results[0][0]
        self.best_model    = results[0][1]
        self.best_hp       = results[0][2]

        self.merge_artifacts(inputs, include=['X_test', 'y_test'])

        # Build the card content
        rows = [[str(r[2]), f"{r[0]:.4f}", "WINNER" if i == 0 else ""] for i, r in enumerate(results)]

        current.card.append(Markdown("# Experiment Results"))
        current.card.append(Markdown(f"**Best accuracy:** {self.best_accuracy:.4f}  \n**Best config:** {self.best_hp}"))
        current.card.append(Table(
            headers=["Hyperparams", "Accuracy", "Note"],
            data=rows
        ))

        self.next(self.end)

    @step
    def end(self):
        print(f"Best: {self.best_hp}  accuracy={self.best_accuracy:.4f}")
        print("View card: python training_flow_cards.py card view join_results")


if __name__ == '__main__':
    ModelTrainingWithCardsFlow()


In [ ]:
!python training_flow_cards.py run


**What just happened?**

- `@card(type='blank')` decorated the `join_results` step — Metaflow generated an HTML report artifact alongside the step's data artifacts.
- **`current.card.append(...)`** adds components to the card during step execution. `Markdown` renders text, `Table` renders a data grid.
- To view the card: `!python training_flow_cards.py card view join_results` — opens the HTML report in a browser.
- In production (Metaflow GUI / Outerbounds), cards appear as visual dashboards in the run timeline without any extra tooling.


## Step 5 · Comparing Runs for Experiment Tracking

Metaflow's Client API provides a simple way to compare metrics across runs — a lightweight alternative to MLflow or W&B for teams already using Metaflow.


In [ ]:
from metaflow import Flow
import pandas as pd

# List all successful runs of the training flow
flow = Flow('ModelTrainingFlow')
rows = []
for run in flow.runs():
    if run.successful:
        try:
            d = run['end'].task.data
            rows.append({
                'run_id':        run.id,
                'finished_at':   run.finished_at,
                'best_accuracy': d.best_accuracy,
                'best_model':    d.best_hp.get('model', 'unknown'),
            })
        except Exception:
            pass  # skip runs without expected artifacts

if rows:
    history = pd.DataFrame(rows).sort_values('run_id')
    print("Experiment history:")
    print(history.to_string(index=False))
else:
    print("Only one run found — run the flow again to see comparison.")


**What just happened?**

- `flow.runs()` iterates over all runs in reverse-chronological order — filtered to `successful` ones.
- We extracted `best_accuracy` and `best_hp` from each run's `end` step artifact.
- **This is Metaflow's built-in experiment tracking**: no extra services needed. Every run's full state is queryable from Python.
- For richer tracking (parallel charts, custom metrics, team dashboards), Metaflow integrates with MLflow and Weights & Biases via the `@wandb_log` and `@mlflow_log` decorators from community extensions.


## Step 6 · Storing the Winning Model for Downstream Inference

Best practice for persisting models in Metaflow:

| Approach | When to use |
|---|---|
| `self.best_model` artifact | Quick local or cloud access via Client API |
| `pickle.dumps` + S3 artifact | Large models — store bytes, not full objects |
| ONNX export | Cross-language inference (Java, C++) |
| Registry (MLflow, BentoML) | When serving requires a registry |


In [ ]:
import pickle
from metaflow import Flow
import numpy as np

# Load the best model from the last training run
run = Flow('ModelTrainingFlow').latest_run
best_model = run['end'].task.data.best_model
X_test     = run['end'].task.data.X_test
y_test     = run['end'].task.data.y_test

# Simulate inference: make predictions on 5 test samples
sample_X = X_test[:5]
sample_y = y_test[:5]
preds = best_model.predict(sample_X)

label_map = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
print("Sample inference results:")
for i, (actual, predicted) in enumerate(zip(sample_y, preds)):
    match = "✓" if actual == predicted else "✗"
    print(f"  [{i}] actual={label_map[actual]:12s}  predicted={label_map[predicted]:12s}  {match}")

# Persist model bytes locally (simulating an S3 upload in production)
model_bytes = pickle.dumps(best_model)
with open('best_model.pkl', 'wb') as f:
    f.write(model_bytes)
print(f"\nModel serialised: {len(model_bytes)/1024:.1f} KB")

# Verify round-trip: load and predict again
loaded_model = pickle.loads(model_bytes)
assert (loaded_model.predict(sample_X) == preds).all(), "Round-trip mismatch!"
print("Round-trip pickle verification: passed")


**What just happened?**

- We loaded the best model artifact from the run and used it for inference — no re-training needed.
- **Pickle round-trip** confirms the serialised model produces identical predictions when deserialised — critical for deployment.
- In production, `model_bytes` would be uploaded to S3 (or another object store) and registered in your model registry; downstream serving code downloads and deserialises it.
- Metaflow's own artifact store already handles serialisation — `self.best_model` is effectively stored this way automatically between steps.


In [ ]:
# Challenge: Extend the training flow with cross-validation scoring
# Your solution here
#
# 1. Write a new flow 'CVTrainingFlow' using %%writefile
# 2. In the train_model step, replace accuracy_score with:
#    from sklearn.model_selection import cross_val_score
#    self.cv_scores = cross_val_score(clf, self.X_train, self.y_train, cv=5)
#    self.cv_mean = float(self.cv_scores.mean())
#    self.cv_std  = float(self.cv_scores.std())
# 3. In join_results, select the best model by cv_mean instead of accuracy
# 4. In end, print a summary including cv_mean ± cv_std for the best config
# 5. Run the flow and compare cv_mean for RF vs LR
#
# Scaffold:
# from sklearn.model_selection import cross_val_score
# self.cv_scores = cross_val_score(clf, self.X_train, self.y_train, cv=5)
# self.cv_mean   = float(self.cv_scores.mean())


---
## Day 11 key concepts recap

| Concept | What to remember |
|---|---|
| `@foreach` | Fan-out over a list — one branch per element; use `self.input` in the foreach step |
| `join(self, inputs)` | Collect all branch results; `inputs` is an iterator of task data |
| `merge_artifacts` | Propagate shared artifacts from branches without manually re-reading them |
| `@card(type='blank')` | Attach a custom HTML report to a step; use `current.card.append(...)` |
| `Markdown`, `Table` | Card components for text and tabular data |
| Client API for tracking | `flow.runs()` iterates all runs; compare `.task.data.metric` across runs |
| Model persistence | Store fitted model as `self.best_model`; serialise to pickle/ONNX for serving |

> **Tip:** `foreach` + `join` is the Metaflow pattern for hyperparameter search — trivially parallelized, artifacts preserved for every run.

---
## What's next
**Day 12** → Production Flows: Scheduling and Deployment — learn `@schedule`, Argo Workflows and AWS Step Functions integration, and production deployment best practices.

Mark Day 11 complete in your [tracker](../index.html).
